# 🧠 Cogito-0.9 API Server
### OpenAI-Compatible REST API | Free Hosting on Kaggle

This notebook:
1. Downloads the Cogito-0.9 GGUF model from HuggingFace
2. Starts a FastAPI server (OpenAI-compatible)
3. Exposes it via a free public tunnel (cloudflared → ngrok → localtunnel → serveo)
4. Manages API keys with rate limiting
5. Keeps the session alive

**⚠️ Kaggle limits**: Sessions run for up to **12 hours** (GPU) or **9 hours** (CPU).
Run this notebook with **GPU T4 x2** for best performance.

## Step 1: Configuration
Set your preferences here. The `ADMIN_KEY` is auto-generated and printed below.

In [ ]:
import os, secrets

# ── Model Config ──────────────────────────────────────────────────────────────
# Choose quantization: 'q4_k_m' (faster, less VRAM) or 'q8_0' (better quality)
QUANT = 'q4_k_m'  # or 'q8_0'

MODEL_REPO = 'ozaa77/Cogito-0.9'
MODEL_FILE = f'cogito-0.9-{QUANT}.gguf'
MODEL_DIR  = '/kaggle/working/models'
MODEL_PATH = f'{MODEL_DIR}/{MODEL_FILE}'

# ── Server Config ─────────────────────────────────────────────────────────────
PORT           = 8000
MAX_CONTEXT    = 4096
N_GPU_LAYERS   = -1     # -1 = all layers on GPU
N_THREADS      = 4
MAX_TOKENS     = 512
RATE_LIMIT_RPM = 30     # requests per minute per key

# ── Tunnel Config ─────────────────────────────────────────────────────────────
NGROK_TOKEN = ''  # Optional: paste your ngrok token for persistent URLs
                  # Get one free at https://dashboard.ngrok.com/get-started/your-authtoken

# ── Keys ──────────────────────────────────────────────────────────────────────
ADMIN_KEY = secrets.token_urlsafe(32)  # Auto-generated admin key
API_KEYS_FILE = '/kaggle/working/api_keys.json'

print('=' * 60)
print('⚙️  Cogito-0.9 API Configuration')
print('=' * 60)
print(f'  Model: {MODEL_FILE}')
print(f'  Port:  {PORT}')
print(f'  GPU Layers: {N_GPU_LAYERS}')
print(f'  Context: {MAX_CONTEXT} tokens')
print()
print(f'  🔑 ADMIN KEY: {ADMIN_KEY}')
print(f'     ⚠️  SAVE THIS KEY! It will not be shown again.')
print('=' * 60)

## Step 2: Install Dependencies

In [ ]:
%%bash
echo '📦 Installing dependencies...'

# Core API server deps
pip install -q fastapi uvicorn[standard] python-multipart huggingface_hub pydantic

# llama-cpp-python with CUDA support
echo '📦 Installing llama-cpp-python with CUDA...'
CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --force-reinstall --no-cache-dir \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 || \
pip install -q llama-cpp-python

echo '✅ Dependencies installed'

## Step 3: Download Model from HuggingFace

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

if Path(MODEL_PATH).exists():
    size_gb = Path(MODEL_PATH).stat().st_size / 1e9
    print(f'✅ Model already exists: {MODEL_PATH} ({size_gb:.2f} GB)')
else:
    print(f'⬇️  Downloading {MODEL_FILE} from HuggingFace...')
    print(f'   This may take a few minutes depending on your internet speed.')
    
    try:
        downloaded = hf_hub_download(
            repo_id=MODEL_REPO,
            filename=MODEL_FILE,
            local_dir=MODEL_DIR,
            local_dir_use_symlinks=False,
        )
        size_gb = Path(downloaded).stat().st_size / 1e9
        print(f'✅ Downloaded: {downloaded} ({size_gb:.2f} GB)')
    except Exception as e:
        print(f'❌ HuggingFace Hub download failed: {e}')
        print('   Trying direct wget...')
        url = f'https://huggingface.co/{MODEL_REPO}/resolve/main/{MODEL_FILE}'
        os.system(f'wget -q --show-progress -O "{MODEL_PATH}" "{url}"')
        if Path(MODEL_PATH).exists():
            print(f'✅ Downloaded via wget')
        else:
            print(f'❌ Download failed! Check your internet connection.')

## Step 4: Write Server Files

In [ ]:
import urllib.request

# Download server files from the repository
BASE_URL = 'https://raw.githubusercontent.com/AlGhozaliRamadhan/Cogito/main/server'
files = ['api_server.py', 'tunnel_manager.py']

for fname in files:
    dest = f'/kaggle/working/{fname}'
    if not os.path.exists(dest):
        try:
            urllib.request.urlretrieve(f'{BASE_URL}/{fname}', dest)
            print(f'✅ Downloaded {fname}')
        except Exception as e:
            print(f'⚠️  Could not download {fname}: {e}')
            print(f'   The file may need to be created manually.')
    else:
        print(f'✅ {fname} already exists')

print('\n📁 Working directory contents:')
for f in os.listdir('/kaggle/working'):
    print(f'  {f}')

## Step 5: Start API Server

In [ ]:
import subprocess, time, threading, os, signal

env = os.environ.copy()
env.update({
    'MODEL_PATH': MODEL_PATH,
    'ADMIN_KEY': ADMIN_KEY,
    'API_KEYS_FILE': API_KEYS_FILE,
    'PORT': str(PORT),
    'MAX_CONTEXT': str(MAX_CONTEXT),
    'N_GPU_LAYERS': str(N_GPU_LAYERS),
    'N_THREADS': str(N_THREADS),
    'MAX_TOKENS_DEFAULT': str(MAX_TOKENS),
    'RATE_LIMIT_RPM': str(RATE_LIMIT_RPM),
})

server_log = '/kaggle/working/server.log'

print('🚀 Starting Cogito-0.9 API Server...')
server_proc = subprocess.Popen(
    ['python', '/kaggle/working/api_server.py'],
    env=env,
    stdout=open(server_log, 'w'),
    stderr=subprocess.STDOUT,
)

# Wait for server to be ready
import socket
for _ in range(60):
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=1):
            print(f'✅ Server is listening on port {PORT}')
            break
    except (ConnectionRefusedError, OSError):
        time.sleep(1)
else:
    print(f'❌ Server did not start in time. Check logs:')
    with open(server_log) as f:
        print(f.read()[-2000:])

print(f'📜 Server log: {server_log}')

## Step 6: Start Public Tunnel

In [ ]:
import subprocess, time, json, urllib.request, os, threading
from pathlib import Path

PUBLIC_URL = None

def try_cloudflared():
    global PUBLIC_URL
    cf_path = '/tmp/cloudflared'
    if not Path(cf_path).exists():
        print('⬇️  Downloading cloudflared...')
        url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
        urllib.request.urlretrieve(url, cf_path)
        os.chmod(cf_path, 0o755)
    
    print('🔄 Starting Cloudflare tunnel...')
    proc = subprocess.Popen(
        [cf_path, 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    
    start = time.time()
    while time.time() - start < 30:
        line = proc.stdout.readline()
        if not line: break
        if 'trycloudflare.com' in line:
            for part in line.split():
                if part.startswith('https://') and 'trycloudflare' in part:
                    PUBLIC_URL = part.strip()
                    return proc, PUBLIC_URL
    return proc, None

def try_ngrok():
    global PUBLIC_URL
    ng_path = '/tmp/ngrok'
    if not Path(ng_path).exists():
        print('⬇️  Downloading ngrok...')
        tgz = '/tmp/ngrok.tgz'
        urllib.request.urlretrieve(
            'https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz', tgz
        )
        os.system(f'tar -xzf {tgz} -C /tmp/')
    
    if NGROK_TOKEN:
        os.system(f'{ng_path} config add-authtoken {NGROK_TOKEN}')
    
    print('🔄 Starting ngrok tunnel...')
    proc = subprocess.Popen(
        [ng_path, 'http', str(PORT)],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    time.sleep(3)
    
    for _ in range(10):
        try:
            with urllib.request.urlopen('http://localhost:4040/api/tunnels', timeout=3) as r:
                data = json.loads(r.read())
                for t in data.get('tunnels', []):
                    if t.get('proto') == 'https':
                        PUBLIC_URL = t['public_url']
                        return proc, PUBLIC_URL
        except: pass
        time.sleep(1)
    return proc, None

# Try tunnels in order
tunnel_proc = None
providers = [
    ('cloudflared', try_cloudflared),
    ('ngrok', try_ngrok),
]

for name, fn in providers:
    try:
        proc, url = fn()
        if url:
            tunnel_proc = proc
            print(f'\n' + '='*60)
            print(f'  ✅ TUNNEL ACTIVE: {name.upper()}')
            print(f'  🌐 PUBLIC URL: {url}')
            print(f'  🔑 ADMIN KEY: {ADMIN_KEY}')
            print(f'  📖 DOCS: {url}/docs')
            print(f'  📊 DASHBOARD: {url}/')
            print('='*60 + '\n')
            break
        else:
            print(f'⚠️  {name} failed, trying next...')
    except Exception as e:
        print(f'⚠️  {name} error: {e}')

if not PUBLIC_URL:
    print('❌ All tunnels failed! The server is still running locally on port', PORT)

## Step 7: Create Your First API Key

In [ ]:
import requests, json

LOCAL = f'http://localhost:{PORT}'

def create_api_key(name, role='user', rate_limit_rpm=30):
    """Create a new API key via the admin endpoint"""
    r = requests.post(
        f'{LOCAL}/v1/admin/keys/create',
        headers={'Authorization': f'Bearer {ADMIN_KEY}'},
        json={'name': name, 'role': role, 'rate_limit_rpm': rate_limit_rpm}
    )
    return r.json()

def list_keys():
    """List all API keys"""
    r = requests.get(
        f'{LOCAL}/v1/admin/keys/list',
        headers={'Authorization': f'Bearer {ADMIN_KEY}'}
    )
    return r.json()

def revoke_key(key):
    """Revoke an API key"""
    r = requests.post(
        f'{LOCAL}/v1/admin/keys/revoke',
        headers={'Authorization': f'Bearer {ADMIN_KEY}'},
        json={'key': key}
    )
    return r.json()

# Create a demo user key
demo_key_data = create_api_key(name='demo-user', role='user', rate_limit_rpm=10)
demo_key = demo_key_data['key']['key']

print('✅ Created demo API key:')
print(f'  Name: demo-user')
print(f'  Key:  {demo_key}')
print(f'  RPM limit: 10')

print('\n📋 All keys:')
all_keys = list_keys()
for k in all_keys.get('keys', []):
    print(f"  [{k['role']:5}] {k['name']:20} | {k['key'][:16]}... | active={k['active']}")

## Step 8: Test the API

In [ ]:
import requests, time

LOCAL = f'http://localhost:{PORT}'

# Wait for model to finish loading
print('⏳ Waiting for model to load...')
for i in range(300):
    try:
        r = requests.get(f'{LOCAL}/health', timeout=3)
        d = r.json()
        if d.get('model_loaded'):
            print('✅ Model is ready!')
            break
        elif d.get('model_loading'):
            print(f'   Loading... ({i+1}s)', end='\r')
        time.sleep(1)
    except:
        time.sleep(1)

# Test chat completion
print('\n🧪 Testing chat completion...')
r = requests.post(
    f'{LOCAL}/v1/chat/completions',
    headers={'Authorization': f'Bearer {demo_key}'},
    json={
        'model': 'cogito-0.9-q4_k_m',
        'messages': [
            {'role': 'system', 'content': 'You are Cogito, a helpful AI assistant.'},
            {'role': 'user', 'content': 'Hello! Introduce yourself in one sentence.'}
        ],
        'max_tokens': 100,
        'temperature': 0.7,
    }
)

if r.status_code == 200:
    response_text = r.json()['choices'][0]['message']['content']
    print(f'\n🤖 Cogito says:')
    print(f'   "{response_text}"')
    print(f'\n✅ API is working!')
else:
    print(f'❌ Error {r.status_code}: {r.text}')

## Step 9: Keep Session Alive (Run this in a separate cell or thread)

In [ ]:
import threading, time, requests, subprocess, datetime
from IPython.display import clear_output, display

LOCAL = f'http://localhost:{PORT}'
MONITOR_INTERVAL = 60  # seconds

def monitor_loop():
    """Monitor and keep alive the server and tunnel"""
    while True:
        try:
            time.sleep(MONITOR_INTERVAL)
            
            # Ping health endpoint
            r = requests.get(f'{LOCAL}/health', timeout=5)
            health = r.json()
            
            # Print status
            uptime = int(health.get('uptime_seconds', 0))
            h, m, s = uptime // 3600, (uptime % 3600) // 60, uptime % 60
            ts = datetime.datetime.now().strftime('%H:%M:%S')
            print(f'[{ts}] 💓 Server alive | uptime={h}h{m}m{s}s | model={'✓' if health['model_loaded'] else '⏳'}')
            
            # Tiny CPU work to prevent idle timeout
            _ = sum(i**2 for i in range(200_000))
            
        except Exception as e:
            print(f'[monitor] Error: {e}')

monitor_thread = threading.Thread(target=monitor_loop, daemon=True)
monitor_thread.start()
print('💓 Monitor started (runs every 60s)')
print('📌 Your API will stay alive as long as this notebook kernel is running.')
print()
print('Summary:')
print(f'  🌐 Public URL:  {PUBLIC_URL}')
print(f'  🔑 Admin Key:   {ADMIN_KEY}')
print(f'  👤 Demo Key:    {demo_key}')
print(f'  📖 Swagger UI:  {PUBLIC_URL}/docs')
print(f'  📊 Dashboard:   {PUBLIC_URL}/')

## 📖 Usage Guide

### Using with OpenAI Python SDK
```python
from openai import OpenAI

client = OpenAI(
    base_url="YOUR_PUBLIC_URL/v1",
    api_key="YOUR_API_KEY",
)

response = client.chat.completions.create(
    model="cogito-0.9-q4_k_m",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)
```

### Creating API Keys (Admin)
```bash
curl -X POST "YOUR_PUBLIC_URL/v1/admin/keys/create" \
  -H "Authorization: Bearer ADMIN_KEY" \
  -H "Content-Type: application/json" \
  -d '{"name": "my-app", "role": "user", "rate_limit_rpm": 20}'
```

### Streaming
```python
for chunk in client.chat.completions.create(..., stream=True):
    print(chunk.choices[0].delta.content or "", end="", flush=True)
```

### Notes
- **Session limit**: Kaggle GPU sessions last up to 12 hours
- **Rate limits**: Default 30 req/min per API key (configurable)
- **Tunnel**: Cloudflare Quick Tunnel URLs change each session
- **Persistence**: To keep keys between sessions, download `api_keys.json`